In [1]:
import pandas as pd
import numpy as np

TARGETS = ["X", "Y"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3", 
    "LSG_1",
    "LSG_2"  
]

results = pd.read_excel("resultados.xlsx")


In [2]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [3]:
df_summary_top10

,dataset,target,mean_r2,std_r2,mean_mse,std_mse
0,Train_1,X,0.997001,0.000180,0.000322,0.000019
1,Train_2,X,0.997959,0.000191,0.000172,0.000016
2,Val,X,0.687331,0.071103,0.029757,0.006767
3,Test_1,X,0.985346,0.001372,0.001527,0.000143
4,Test_2,X,0.983242,0.007611,0.001376,0.000625
5,Test_3,X,0.952425,0.005528,0.004359,0.000506
6,Train_1,Y,0.968014,0.011411,0.002675,0.000954
7,Train_2,Y,0.967186,0.003986,0.003515,0.000427
8,Val,Y,0.960410,0.010792,0.002943,0.000802
9,Test_1,Y,0.989533,0.001693,0.000856,0.000138


In [4]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
   Target  Dataset                                    Best_Model      Neurons  \
0       X  Train_1       model_arch16-8_r0.9_Ld0.3_Lp0.7_seed660      [16, 8]   
1       Y  Train_1      model_arch8-4_r0.01_Ld0.7_Lp0.3_seed4458       [8, 4]   
2       X  Train_2   model_arch32-16-8_r0.9_Ld0.3_Lp0.7_seed7533  [32, 16, 8]   
3       Y  Train_2       model_arch128_r0.9_Ld0.7_Lp0.3_seed4458        [128]   
4       X      Val       model_arch8-4_r0.9_Ld0.3_Lp0.7_seed3980       [8, 4]   
5       Y      Val        model_arch16_r0.9_Ld0.7_Lp0.3_seed7533         [16]   
6       X   Test_1   model_arch16-8-4_r0.01_Ld0.3_Lp0.7_seed7034   [16, 8, 4]   
7       Y   Test_1       model_arch16_r0.01_Ld0.7_Lp0.3_seed7034         [16]   
8       X   Test_2        model_arch16_r0.9_Ld0.7_Lp0.3_seed7034         [16]   
9       Y   Test_2  model_arch264-128_r0.01_Ld0.7_Lp0.3_seed3980   [264, 128]   
10      X   Test_3    model_arch16-8-4_r0.9_Ld0.7_Lp0.3_seed703